In [ ]:
from importlib import reload

from occhio.distributions import SparseUniform, DAGRandomWalkToRoot
from occhio.model_grid import ModelGrid, Axis
from occhio.toy_model import ToyModel
from occhio.autoencoder import TiedLinearRelu
from occhio.visualization import (
    plot_embedding,
    plot_phase_change,
    plot_dynamic_scatter,
    plot_geometry,
    plot_phase_change_multi,
)
import torch
import numpy as np

In [ ]:
dag_formations = [
    np.array(
        [
            [0.0, 0.0, 1.0],
            [0.0, 0.0, 1.0],
            [0.0, 0.0, 0.0],
        ]
    ),
    np.array(
        [
            [0.0, 1.0, 1.0],
            [0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0],
        ]
    ),
    np.array(
        [
            [0.0, 1.0, 0.0],
            [0.0, 0.0, 1.0],
            [0.0, 0.0, 0.0],
        ]
    ),
    np.array(
        [
            [0.0, 1.0, 1.0],
            [0.0, 0.0, 1.0],
            [0.0, 0.0, 0.0],
        ]
    ),
]

In [ ]:
device = "cpu"

N_FEATURES = 10
N_HIDDEN = 2

axis1 = Axis(label="p_edge", values=np.array([0.5, 1.0, 3.0]) / N_FEATURES)
axis2 = Axis(label="beta", values=[0.1, 0.5, 0.9])


def create_model(params):
    generator = torch.Generator(device=device).manual_seed(8)
    p_edge = params["p_edge"]
    beta = params["beta"]
    importance = 0.95

    return ToyModel(
        distribution=DAGRandomWalkToRoot(
            N_FEATURES, p_edge=p_edge, beta=beta, generator=generator
        ),
        ae=TiedLinearRelu(N_FEATURES, N_HIDDEN, generator=generator),
        importances=importance ** torch.arange(N_FEATURES),
        device=device,
    )

In [ ]:
grid = ModelGrid(
    create_model,
    axes=[axis1, axis2],
)
grid.fit(batch_size=512, n_epochs=25_000)

In [ ]:
plot_embedding(grid)

### Observations
So in general the representation becomes more superpositional as `p_edge` and `beta` become smaller.
This makes sense to me in the following way:
- If `p_edge` is smaller the features become less correlated (but sparse)
- If `beta` becomes smaller the magnitudes become less correlated


The angles between vectors are more wonky, and the model seems to like to represent things in 90 degree angles if it can.

Similarly, parents like to "shadow" their children (or the other way around)

# Phase Change

In [ ]:
device = "cpu"

N_FEATURES = 10
N_HIDDEN = 2

axis1 = Axis(label="E(neighbors)", values=torch.linspace(start=0, end=4, steps=10))
axis2 = Axis(label="beta", values=torch.linspace(start=0, end=1, steps=10))


def create_model(params):
    generator = torch.Generator(device=device).manual_seed(8)
    e_neighbors = params["E(neighbors)"]
    beta = params["beta"]
    importance = 0.95

    return ToyModel(
        distribution=DAGRandomWalkToRoot(
            N_FEATURES, p_edge=e_neighbors / N_FEATURES, beta=beta, generator=generator
        ),
        ae=TiedLinearRelu(N_FEATURES, N_HIDDEN, generator=generator),
        importances=importance ** torch.arange(N_FEATURES),
        device=device,
    )

In [ ]:
grid = ModelGrid(
    create_model=create_model,
    axes=[axis1, axis2],
)

In [ ]:
grid.fit(batch_size=256, n_epochs=8_000)

In [ ]:
fig = plot_phase_change_multi(grid, up_to=5, max_cols=3)
fig.update_layout(height=600)
fig.show()

# Learning Dynamics

In [ ]:
N_FEATURES = 10
N_HIDDEN = 2


def my_hook(hook_data):
    return hook_data["epoch"], hook_data["tm"].ae.W.detach().numpy().copy()


# Another hook here!
def feat_dim_and_interference(hook_data):
    return hook_data["epoch"], torch.stack(
        [
            hook_data["tm"].feature_dimensionalities,
            hook_data["tm"].total_feature_interferences,
        ]
    )


generator = torch.Generator(device=device).manual_seed(8)
p_edge = 2.0 / N_FEATURES
beta = 0.9
importance = 0.95

tm = ToyModel(
    distribution=DAGRandomWalkToRoot(
        N_FEATURES, p_edge=p_edge, beta=beta, generator=generator
    ),
    ae=TiedLinearRelu(N_FEATURES, N_HIDDEN, generator=generator),
    importances=importance ** torch.arange(N_FEATURES),
    device=device,
)
losses, hook_returns = tm.fit(
    40_000,
    batch_size=1024,
    verbose=False,
    hooks=[my_hook, feat_dim_and_interference],
    hook_freq=250,
    learning_rate=3e-4,
    weight_decay=0.05,
)

In [ ]:
fig = plot_dynamic_scatter(losses, hook_returns[1], loss_stride=20)
fig.update_layout(height=int(800))
fig.show()

# Next Experiment

In [ ]:
device = "cpu"

N_FEATURES = 400
N_HIDDEN = 30


axis = Axis(label="Density", values=torch.logspace(-2, 0, 32))


def create_model(params):
    generator = torch.Generator(device=device).manual_seed(8)
    importance = 0.999  # params["Importance"]
    p_active = params["Density"]

    return ToyModel(
        distribution=SparseUniform(
            N_FEATURES, p_active, device=device, generator=generator
        ),
        importances=importance ** torch.arange(N_FEATURES),
        ae=TiedLinearRelu(N_FEATURES, N_HIDDEN, device=device, generator=generator),
    )

In [ ]:
grid = ModelGrid(
    create_model,
    axes=[axis],
)

In [ ]:
grid.fit(batch_size=256, n_epochs=10_000)

In [ ]:
plot_geometry(grid)